In [ ]:
import cv2
import numpy as np
import os

# ============================================
# NOMES DAS IMAGENS DE ENTRADA
# ============================================

# caio_sem_avatar.jpg
# caio_com_avatar.jpg
# edson_sem_avatar.jpg
# edson_com_avatar.jpg
# nicolas_sem_avatar.jpg
# nicolas_com_avatar.jpg

# ============================================
# CONFIGURAÇÕES
# ============================================

# Lista de pessoas (nomes dos arquivos sem extensão)
pessoas = ['caio', 'edson', 'nicolas']

# Variações: 'sem_avatar' e 'com_avatar'
versoes = ['sem_avatar', 'com_avatar']

# Extensão das imagens (padrão .jpg)
extensao = '.jpg'

# --- Sobel: tamanhos de kernel a testar ---
# Valores possíveis: 1, 3, 5, 7 (ímpares)
sobel_kernels = [3, 5]  # Altere aqui para testar outros valores

# --- Canny: lista de dicionários com limiares ---
# Cada dicionário deve conter:
#   'threshold1': limiar inferior
#   'threshold2': limiar superior
#   'desc': descrição curta (usada no nome do arquivo)
# Altere os valores abaixo para testar diferentes combinações
canny_params = [
    {'threshold1': 30, 'threshold2': 90, 'desc': 'baixo'},
    {'threshold1': 100, 'threshold2': 200, 'desc': 'medio'},
    {'threshold1': 200, 'threshold2': 400, 'desc': 'alto'}
    # Adicione mais variações se desejar
]

# ============================================
# PROCESSAMENTO (arquivos na mesma pasta)
# ============================================

print("Iniciando processamento...")

for pessoa in pessoas:
    for versao in versoes:
        # Monta o nome do arquivo de entrada (ex: caio_sem_avatar.jpg)
        nome_entrada = f"{pessoa}_{versao}{extensao}"

        # Verifica se o arquivo existe no diretório atual
        if not os.path.exists(nome_entrada):
            print(f"Arquivo não encontrado: {nome_entrada}. Pulando...")
            continue

        # Carregar imagem
        img = cv2.imread(nome_entrada)
        if img is None:
            print(f"Erro ao ler: {nome_entrada}")
            continue

        # Converter para escala de cinza
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        print(f"Processando: {nome_entrada}")

        # ---------- Laplaciano ----------
        laplacian = cv2.Laplacian(gray, cv2.CV_64F)
        laplacian = np.uint8(np.abs(laplacian))
        nome_saida = f"laplacian_{pessoa}_{versao}.jpg"
        cv2.imwrite(nome_saida, laplacian)

        # ---------- Sobel (X e Y) para cada kernel ----------
        for k in sobel_kernels:
            # Sobel X
            sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=k)
            sobelx = np.uint8(np.abs(sobelx))
            nome_saida = f"sobelx_k{k}_{pessoa}_{versao}.jpg"
            cv2.imwrite(nome_saida, sobelx)

            # Sobel Y
            sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=k)
            sobely = np.uint8(np.abs(sobely))
            nome_saida = f"sobely_k{k}_{pessoa}_{versao}.jpg"
            cv2.imwrite(nome_saida, sobely)

        # ---------- Canny para cada conjunto de parâmetros ----------
        for params in canny_params:
            edges = cv2.Canny(gray, params['threshold1'], params['threshold2'])
            nome_saida = f"canny_{params['desc']}_{pessoa}_{versao}.jpg"
            cv2.imwrite(nome_saida, edges)

print("Processamento concluído!")